# Recent frequency detectors under the four-axis protocol

Faithful re-implementations of three recent **frequency-based** adversarial detectors (HFAmp ~ Li 2024; DCTspec ~ Huang 2024 / Frank 2020; DFTspec), each with a **trained** logistic classifier on its spectral feature, evaluated on our protocol (pristine / +hard-negative / operational TPR) for CIFAR-10 (upscaled x7) and ImageNet (native 224).

**Run:** in `/home/jupyter/SCAN`, Run All. Reuses `mixed_dataset.pkl` caches (`cifar10_detector_results`, `imagenet_val_detector_results_eps8`) and the same hard-negative construction (noise8 + JPEG75 + blur1) as the centerpiece harness. Output: `recent_detectors_results.json` + a summary table.

**Hypothesis:** recent frequency detectors also inflate on upscaled data and collapse under hard negatives / native resolution — i.e., the artifact is general to the frequency family, answering the "naive detectors" critique.

In [ ]:
# ============ [PREAMBLE] self-contained helpers (verbatim from centerpiece harness) ============
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']; res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406]; IMGNET_STD=[0.229,0.224,0.225]
def make_pp(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1).to(device); std=torch.tensor(s).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    cfg={'CIFAR-10':('resnet50_cifar10_finetuned.pt',10),'CIFAR-100':('resnet50_cifar100_finetuned.pt',100),
         'SVHN':('resnet50_svhn_finetuned.pt',10),'TinyImageNet':('resnet50_tinyimagenet_finetuned.pt',200)}
    if ds in cfg:
        ck=(_find(cfg[ds][0]) or [None])[0]; m=models.resnet50(weights=None); m.fc=nn.Linear(2048,cfg[ds][1])
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict'])
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75):
    return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet',p)
    return out
def auc_ci(neg,pos,B=2000,seed=SEED):
    neg=np.asarray(neg);pos=np.asarray(pos)
    base=roc_auc_score(np.r_[np.zeros(len(neg)),np.ones(len(pos))],np.r_[neg,pos])
    rng=np.random.RandomState(seed); b=[]
    for _ in range(B):
        nb=neg[rng.randint(0,len(neg),len(neg))]; pb=pos[rng.randint(0,len(pos),len(pos))]
        b.append(roc_auc_score(np.r_[np.zeros(len(nb)),np.ones(len(pb))],np.r_[nb,pb]))
    return float(base),float(np.percentile(b,2.5)),float(np.percentile(b,97.5))
def tpr_at_fpr(neg,pos,fpr):
    thr=np.quantile(np.asarray(neg),1-fpr); return float((np.asarray(pos)>=thr).mean())
def half(n, seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]
def _gray255(b):
    w=torch.tensor([0.299,0.587,0.114],device=b.device).view(1,3,1,1)
    return (b*w).sum(1)
print('[PREAMBLE] helpers ready; device =', device)


In [ ]:
# ============ [RECENT FREQUENCY DETECTORS] faithful re-implementations (vector features + trained classifier) ============
# Each recent method reduces to a high-frequency / spectral representation fed to a trained binary classifier.
# We implement the CORE mechanism of three published families and train a logistic classifier (clean vs adversarial),
# exactly the "learned frequency detector" the reviewer asks us to compare against.
#   HFAmp   ~ Li (2024)  high-frequency amplification: multi-scale high-pass residual energy
#   DCTspec ~ Huang (2024) / Frank (2020): 8x8 block-DCT mean-magnitude spectrum (64-D)
#   DFTspec ~ frequency-domain family: radial DFT power profile (32-D)
from scipy.fft import dctn

def feat_hfamp_vec(b, sigmas=(0.5,1.0,2.0)):
    # multi-scale high-pass residual energy (mean|.|, std, energy) per scale -> 9-D
    feats=[]
    for s in sigmas:
        rg=_gray255((b-gb(b,s))).abs()            # [N,H,W]
        feats += [rg.flatten(1).mean(1), rg.flatten(1).std(1), (rg**2).flatten(1).mean(1)]
    return torch.stack(feats,1).detach().cpu().numpy()          # [N,9]

def feat_dctspec_vec(b, B=8):
    # 8x8 block-DCT, mean |coefficient| over all blocks -> 64-D spectrum
    g=_gray255(b).detach().cpu().numpy(); N,H,W=g.shape
    Hc,Wc=(H//B)*B,(W//B)*B; nby,nbx=Hc//B,Wc//B
    blk=g[:,:Hc,:Wc].reshape(N,nby,B,nbx,B).transpose(0,1,3,2,4)  # [N,nby,nbx,B,B]
    d=np.abs(dctn(blk,axes=(-2,-1),norm='ortho')).mean(axis=(1,2))  # [N,B,B]
    return d.reshape(N,B*B).astype(np.float64)                    # [N,64]

def feat_dftspec_vec(b, nbins=32):
    # radial DFT power profile (log) -> 32-D
    g=_gray255(b).detach().cpu().numpy(); N,H,W=g.shape
    yy,xx=np.mgrid[0:H,0:W]; cy,cx=H/2.0,W/2.0
    r=np.sqrt(((yy-cy)/cy)**2+((xx-cx)/cx)**2); rb=np.clip((r/r.max()*nbins).astype(int),0,nbins-1)
    masks=[rb==k for k in range(nbins)]
    out=np.zeros((N,nbins))
    for n in range(N):
        sp=np.abs(np.fft.fftshift(np.fft.fft2(g[n])))
        for k,m in enumerate(masks): out[n,k]=sp[m].mean() if m.any() else 0.0
    return np.log1p(out)                                          # [N,32]

DETECTORS={'HFAmp (Li 2024, HF amplification)':feat_hfamp_vec,
           'DCTspec (DCT spectrum, Huang 2024/Frank 2020)':feat_dctspec_vec,
           'DFTspec (radial DFT profile)':feat_dftspec_vec}

def batched_feat(fn, X, bs=64):
    out=[]
    for i in range(0,len(X),bs): out.append(fn(X[i:i+bs].to(device)))
    return np.concatenate(out,0)
print('[DETECTORS] 3 recent frequency detectors ready:', list(DETECTORS))


In [ ]:
# ============ [4-AXIS PROTOCOL] train each detector, evaluate on hard-neg / native / operational ============
# For a TRAINED detector we split clean AND adversarial into train/test (no leakage):
#   train logistic on (clean_train=0, adv_train=1); evaluate scores on held-out test.
#   pristine  : clean_test (neg) vs adv_test (pos)
#   +hard-neg : clean_test + benign hard-negatives (noise8+JPEG75+blur1) (neg) vs adv_test (pos)
#   operational TPR@FPR: threshold set on the hard-negative (operational) negative set.
# CIFAR-10 pkl = upscaled x7 regime; ImageNet pkl = native 224 regime.
N_CLEAN=500; FPRS=[0.01,0.05,0.10]
MX=find_mixed(); assert MX, 'no mixed_dataset.pkl found'
print('using caches:', MX)
RESULTS_RECENT={}

for ds,pkl in MX.items():
    regime='upscaled x7' if ds=='CIFAR-10' else 'native 224'
    mixed=pickle.load(open(pkl,'rb'))
    clean=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
    adv  =[to224(im).cpu() for (im,lb,atk) in mixed if atk!='clean']
    rng=np.random.RandomState(SEED)
    clean=[clean[i] for i in rng.permutation(len(clean))[:N_CLEAN]]
    # split clean and adv into train/test halves (no leakage for the trained classifier)
    cids=np.arange(len(clean)); rng.shuffle(cids); c_tr,c_te=cids[:len(cids)//2],cids[len(cids)//2:]
    aids=np.arange(len(adv));   rng.shuffle(aids); a_tr,a_te=aids[:len(aids)//2],aids[len(aids)//2:]
    Xc_tr=torch.cat([clean[i] for i in c_tr],0); Xc_te=torch.cat([clean[i] for i in c_te],0)
    Xa_tr=torch.cat([adv[i]   for i in a_tr],0); Xa_te=torch.cat([adv[i]   for i in a_te],0)
    # benign hard negatives from the clean TEST half (matched-budget noise + JPEG q75 + blur sigma1)
    hn_noise=(Xc_te+torch.randn_like(Xc_te)*8.0).clamp(0,255)
    hn_jpeg =jpeg_batch(Xc_te.to(device)).cpu()
    hn_blur =gb(Xc_te.to(device),1.0).clamp(0,255).cpu()
    Xhn=torch.cat([hn_noise,hn_jpeg,hn_blur],0)

    RESULTS_RECENT[ds]={'regime':regime}
    for name,fn in DETECTORS.items():
        Ftr=np.r_[batched_feat(fn,Xc_tr), batched_feat(fn,Xa_tr)]
        ytr=np.r_[np.zeros(len(Xc_tr)), np.ones(len(Xa_tr))]
        sc=StandardScaler().fit(Ftr)
        clf=LogisticRegression(max_iter=2000,C=1.0).fit(sc.transform(Ftr),ytr)
        score=lambda X: clf.decision_function(sc.transform(batched_feat(fn,X)))
        s_cte=score(Xc_te); s_ate=score(Xa_te); s_hn=score(Xhn)
        neg_p=s_cte; pos=s_ate; neg_hn=np.r_[s_cte,s_hn]
        a_p,lo_p,hi_p=auc_ci(neg_p,pos); a_h,lo_h,hi_h=auc_ci(neg_hn,pos)
        RESULTS_RECENT[ds][name]={
            'pristine_auroc':round(a_p,4),'pristine_ci':[round(lo_p,4),round(hi_p,4)],
            'hardneg_auroc':round(a_h,4),'hardneg_ci':[round(lo_h,4),round(hi_h,4)],
            'delta_auc':round(a_h-a_p,4),
            'tpr_pristine':{f'{int(f*100)}%':round(tpr_at_fpr(neg_p,pos,f),4) for f in FPRS},
            'tpr_operational':{f'{int(f*100)}%':round(tpr_at_fpr(neg_hn,pos,f),4) for f in FPRS},
        }
        print(f'[{ds:8s} {regime:11s}] {name[:34]:34s} pristine={a_p:.3f}  +hardneg={a_h:.3f}  dAUC={a_h-a_p:+.3f}  opTPR@5={RESULTS_RECENT[ds][name]["tpr_operational"]["5%"]:.3f}')

json.dump(RESULTS_RECENT, open('recent_detectors_results.json','w'), indent=2)
print('\nsaved recent_detectors_results.json')


In [ ]:
# ============ [SUMMARY] appendix-ready table + interpretation ============
import pandas as pd
rows=[]
for ds in RESULTS_RECENT:
    for name,v in RESULTS_RECENT[ds].items():
        if name=='regime': continue
        rows.append({'Dataset':f'{ds} ({RESULTS_RECENT[ds]["regime"]})','Detector':name.split(' (')[0],
                     'pristine AUROC':v['pristine_auroc'],'+hard-neg AUROC':v['hardneg_auroc'],
                     'dAUC':v['delta_auc'],'oper TPR@5%FPR':v['tpr_operational']['5%']})
df=pd.DataFrame(rows)
print(df.to_string(index=False))
print('\n% ---- paste-ready LaTeX rows (Detector & pristine & +hardneg & dAUC & opTPR@5) ----')
for _,r in df.iterrows():
    print(f"  {r['Detector']:8s} & {r['pristine AUROC']:.3f} & {r['+hard-neg AUROC']:.3f} & {r['dAUC']:+.3f} & {r['oper TPR@5%FPR']:.3f} \\\\  % {r['Dataset']}")
print('\nEXPECTED PATTERN (thesis): on upscaled CIFAR-10 all three recent frequency detectors reach near-perfect')
print('pristine AUROC but drop sharply under benign hard negatives; on native ImageNet they sit near chance.')
print('=> recently published frequency detectors fall for the SAME upscaling artifact, not just our HF-Energy.')
